# Medical Image Diagnosis with MRI: DICOM, the Lumbar Spine, and Evaluating MedGemma
**AI in Healthcare · Unit III · Lecture**

A person with long-standing low back pain, and pain that runs down one leg, is referred for a lumbar spine MRI. The scanner produces about a hundred images in three separate series. A radiologist (a doctor who interprets medical images) scrolls through them, finds each disc level, and grades how narrow the spaces around the nerves have become. That report then helps the referring clinician decide what to do next.

Can a medical vision–language model read those images and produce the same grades? For an engineer, the more important question is: **how would we find out, and how could we be fooled?**

In this lecture you will learn enough anatomy and MRI physics to understand the task, see how real MRI files are stored in DICOM format and displayed, and plan a small, honest evaluation of MedGemma against expert labels for one patient. In [Lab 9](../../labs/lab-09-lumbar-mri-dicom-medgemma.ipynb) you carry out that evaluation.

> **Educational use only.** The images are de-identified research data from the RSNA 2024 Lumbar Spine Degenerative Classification dataset. Model outputs in this notebook are not clinical advice. Results from one patient cannot establish how well any model performs.

**Prerequisites:** Python, NumPy, pandas, and matplotlib; supervised classification; confusion-matrix metrics (sensitivity, specificity, PPV, NPV). No medical training is assumed.

**Companions:** [Medical foundation models lecture](../../../module2/lectures/session-05-medical-foundation-models.ipynb) · [Optional lab — Evaluating medical foundation models](../../../module2/labs/lab-06-medical-foundation-models.ipynb) · [Skin and tissue image demonstration](session-02-medical-image-analysis.ipynb) · [Lab 9 — hands-on DICOM loading and MedGemma evaluation](../../labs/lab-09-lumbar-mri-dicom-medgemma.ipynb)

## Learning objectives

By the end, you should be able to:

1. Describe the lumbar spine structures involved in degenerative stenosis, and name the MRI view used to grade each of the five RSNA conditions.
2. Explain how T1-weighted, T2-weighted, and STIR images differ, and what sagittal and axial planes show.
3. Explain how DICOM organizes images into patients, studies, series, and instances, and distinguish identifiers from image content.
4. Use DICOM geometry tags to find an image's plane, physical size, and the patient's left and right.
5. Explain how window/level changes what is visible, and what MedGemma actually receives.
6. Describe how to score model labels against a reference standard and a baseline, and explain why one patient cannot establish clinical performance.

## Contents

- [About the data used in this lecture](#About-the-data-used-in-this-lecture)

1. [The clinical problem: back pain, nerves, and narrowing](#1.-The-clinical-problem:-back-pain,-nerves,-and-narrowing)
2. [MRI in brief](#2.-MRI-in-brief)
3. [DICOM: how medical images are stored](#3.-DICOM:-how-medical-images-are-stored)
4. [Geometry: from pixels to millimetres and patient directions](#4.-Geometry:-from-pixels-to-millimetres-and-patient-directions)
5. [Displaying MRI: window/level, and what a model actually sees](#5.-Displaying-MRI:-window/level,-and-what-a-model-actually-sees)
6. [Two ways to build imaging AI](#6.-Two-ways-to-build-imaging-AI)
7. [How we will evaluate MedGemma](#7.-How-we-will-evaluate-MedGemma)


[Key ideas](#Key-ideas) · [References and further reading](#References-and-further-reading)

The hands-on part is [Lab 9](../../labs/lab-09-lumbar-mri-dicom-medgemma.ipynb).

## Setup

**Software.** Python 3.10+ with `numpy`, `pandas` (2.1+), and `pydicom`. This notebook does not install anything. If something is missing, install it in your course environment, for example `pip install pydicom`.

**Data.** Your instructor provides the folder `data/rsna_study_100206310/`, next to this notebook (about 16 MB). It holds one study from the RSNA 2024 Lumbar Spine Degenerative Classification training set: three MRI series (96 DICOM files), plus the three label files filtered to this study. The lecture reads one DICOM file from it (§3–4); [Lab 9](../../labs/lab-09-lumbar-mri-dicom-medgemma.ipynb) uses the whole folder. The full dataset is on the [Kaggle competition page](https://www.kaggle.com/competitions/rsna-2024-lumbar-spine-degenerative-classification). It needs a Kaggle account and acceptance of the competition rules, and its authors make it available for non-commercial use. Do not redistribute the files outside the class.

**Compute.** Runs on an ordinary laptop CPU in seconds. No model is called.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from IPython.display import display

print("pydicom", pydicom.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

pydicom 3.0.2 | pandas 2.2.3 | numpy 1.26.4


In [2]:
# ---- Data location (relative to this notebook) ----
STUDY_ID = "100206310"
DATA_ROOT = Path("data/rsna_study_100206310")
IMAGE_ROOT = DATA_ROOT / "train_images" / STUDY_ID

assert IMAGE_ROOT.exists(), f"Image folder not found: {IMAGE_ROOT.resolve()} (see the Data note above)"
print("Images:", IMAGE_ROOT)

Images: data/rsna_study_100206310/train_images/100206310


---
## About the data used in this lecture

Everything in this lecture and in Lab 9 comes from **one patient**: a single lumbar spine MRI study (`study_id 100206310`) from the training set of the [RSNA 2024 Lumbar Spine Degenerative Classification](https://www.kaggle.com/competitions/rsna-2024-lumbar-spine-degenerative-classification) dataset. The images are real but de-identified. The terms used below (sagittal, axial, T1, T2, DICOM) are explained in sections 2–4.

**Images: 3 series, 96 DICOM files, about 16 MB**

| Series | View | Slices | Image size | Pixel size | Slice thickness |
|---|---|---|---|---|---|
| Sagittal T2/STIR | Side view, slicing from the patient's right to left | 18 | 462 × 448 pixels (about 247 × 240 mm) | 0.54 mm | 4 mm |
| Sagittal T1 | Side view, slicing from the patient's right to left | 18 | 512 × 512 pixels (about 350 × 350 mm) | 0.68 mm | 4 mm |
| Axial T2 | Cross-sections, from the head end downwards, in five tilted blocks | 60 | 320 × 320 pixels (160 × 160 mm) | 0.5 mm | 3.5 mm |

All images are 12-bit greyscale. The files were de-identified, so their headers are sparse. For example, `Modality` and the patient's name are absent, and `PatientID` is simply the study number.

**Labels: three CSV files, filtered to this study**

| File | Contents for this patient |
|---|---|
| `train.csv` | 25 severity grades: 5 conditions × 5 disc levels (L1/L2 to L5/S1), each Normal/Mild, Moderate, or Severe |
| `train_label_coordinates.csv` | 25 points showing the slice and pixel where the radiologist made each grading |
| `train_series_descriptions.csv` | The name of each of the 3 series (for example, "Axial T2") |

This patient has substantial degenerative disease: most of the 25 grades are Moderate or Severe, and L4/L5 is the most affected level. You will count the grades yourself in [Lab 9](../../labs/lab-09-lumbar-mri-dicom-medgemma.ipynb), Exercise 1.

**Saved model responses.** `medgemma_runs/` holds MedGemma's replies to the 17 prompts used in Lab 9, so the lab runs without a GPU.

**What this data can and cannot show**

- The grades are **one radiologist's reading**: training-set studies were annotated only once ([Richards et al., 2026](https://pubs.rsna.org/doi/10.1148/ryai.250480)). Treat them as a reference standard, not certain truth.
- One patient can **reveal how a model behaves and fails**. It cannot measure accuracy, because the 25 labels share one person's anatomy, one scanner, and one annotator.
- The dataset is available for non-commercial use under the Kaggle competition rules. Keep these files within the class.

## 1. The clinical problem: back pain, nerves, and narrowing

### 1.1 What is in the lower back?

The **spine** is a stack of bones called **vertebrae**. The lower back is the **lumbar** region, with five vertebrae named **L1 to L5** from top to bottom. Below L5 is the **sacrum**, whose top segment is called **S1**.

Between neighbouring vertebrae sits an **intervertebral disc**: a cushion with a water-rich, gel-like centre and a tough fibrous outer ring. A disc **level** is named after the two bones around it. For example, **L4/L5** is the disc between L4 and L5. The five lumbar levels are L1/L2, L2/L3, L3/L4, L4/L5, and L5/S1.

```text
 Side view (sagittal), simplified

   SUPERIOR (towards the head)
      ┌──────┐
      │  L1  │   ▒▒  ← spinal canal (behind the bones)
      └──────┘   ▒▒
       ====  L1/L2 disc
      ┌──────┐   ▒▒
      │  L2  │   ▒▒
      └──────┘   ▒▒
       ====  L2/L3        … and so on down to …
      ┌──────┐   ▒▒
      │  L5  │   ▒▒
      └──────┘   ▒▒
       ====  L5/S1 disc
      ┌──────┐
      │  S1  │  (sacrum)
      └──────┘
   INFERIOR (towards the feet)
   ANTERIOR (front) ◄────► POSTERIOR (back)
```

### 1.2 Where do the nerves run?

Behind the discs and vertebral bodies is the **spinal canal**. It contains the **thecal sac**, a membrane tube filled with **cerebrospinal fluid (CSF)** in which nerve roots float. In the lower back, the spinal cord itself has usually ended (around L1–L2). Below that, the canal holds a bundle of nerve roots called the **cauda equina** ("horse's tail").

At each level, a pair of nerve roots leaves the spine, one on each side. On its way out, a root passes through two narrow spaces:

- the **lateral recess** (also called the **subarticular zone**): a gutter at the side of the canal, under the facet joint;
- the **neural foramen**: a window between neighbouring vertebrae through which the root exits. It is normally cushioned by fat.

```text
 Cross-section (axial) at one disc level, simplified.
 Drawn in radiological convention: viewed from the feet, so the
 PATIENT'S RIGHT is on the LEFT of the picture.

                     ANTERIOR (front)
               ┌─────────────────────────┐
               │      intervertebral     │
               │           disc          │
               └─────────────────────────┘
   right       ┌──┐  ┌───────────────┐  ┌──┐       left
   neural ◄──  │  │  │  thecal sac   │  │  │  ──►  neural
   foramen     └──┘  │ (CSF + nerve  │  └──┘       foramen
          right      │    roots)     │     left
          lateral ►  └───────────────┘  ◄ lateral
          recess    facet  ═══════  facet   recess
                    joint  ligamentum joint
                           flavum
                     POSTERIOR (back)
   Patient's RIGHT  ◄──────────────────►  Patient's LEFT
```

### 1.3 What goes wrong: degenerative stenosis

With age and wear, discs lose water and height and may bulge backwards. The **facet joints** (small joints at the back of each vertebra) can enlarge, and the **ligamentum flavum** (the ligament forming the back wall of the canal) can thicken. Each change takes up space.

**Stenosis** means narrowing. Degenerative changes can narrow the central canal, a lateral recess, or a foramen. If the narrowing squeezes a nerve root, the person may feel pain, numbness, or weakness in the leg.

Narrowing on an image does not prove that it causes symptoms. Degenerative findings are common in people with no back pain and become more common with age. In one systematic review, disc degeneration was reported in 37% of asymptomatic 20-year-olds and 96% of asymptomatic 80-year-olds ([Brinjikji et al., 2015](https://www.ajnr.org/content/36/4/811)).

So the radiologist grades **anatomy**. The clinician then combines that grading with the person's symptoms and examination. The AI task in this lecture and Lab 9 is anatomical grading, which is far narrower than "diagnosing the cause of back pain".

### 1.4 The five conditions in the RSNA dataset

The RSNA 2024 dataset contains MRI studies from 2,697 patients at 8 institutions in 6 countries. Volunteer neuroradiologists and musculoskeletal radiologists graded five conditions at each of the five levels ([Richards et al., 2026](https://pubs.rsna.org/doi/10.1148/ryai.250480)). The table also shows which series the annotators used to grade each condition.

| Condition | Where the narrowing is | Sides | Series used by annotators | What narrowing looks like (simplified) |
|---|---|---|---|---|
| **Spinal canal stenosis** | Central canal | One (midline) | Sagittal "T2-like" | The bright CSF around the nerve roots is squeezed out |
| **Neural foraminal narrowing** | Foramen | Left and right | Sagittal T1 | The bright fat around the exiting root is reduced |
| **Subarticular stenosis** | Lateral recess | Left and right | Axial T2 | The recess narrows and the root is crowded |

That gives **5 condition columns × 5 levels = 25 labels per study**. Annotators used a 4-point scale (normal, mild, moderate, severe), which was then collapsed into three classes by merging normal and mild: **Normal/Mild, Moderate, Severe**.

The classes are imbalanced. Across all spinal canal grades in the dataset, 85.4% were Normal/Mild, 8.8% Moderate, and 5.9% Severe.

**How good is the "ground truth"?** Training-set studies were **annotated once**, by one radiologist per condition. Test-set studies were re-annotated until two annotators agreed. The dataset authors note that grading degenerative disease is inherently subjective ([Richards et al., 2026](https://pubs.rsna.org/doi/10.1148/ryai.250480)). Our patient comes from the training set, so its labels are **one expert's reading**. That makes them a **reference standard**, not certain truth.

### 1.5 Clinical purpose versus computational task

Before choosing a model, separate the real-world purpose from the small, testable task delegated to software.

| Question | Answer for this lecture and Lab 9 |
|---|---|
| Clinical purpose | Support consistent grading of lumbar stenosis in a radiology report |
| Not the purpose | Diagnosing the cause of pain, or deciding on treatment or surgery |
| Intended user | A radiologist who reviews every output |
| Decision time | After the scan is complete, before the report is signed |
| Unit of analysis | One **study**: one MRI visit, here one patient |
| Input | Three MRI series (sagittal T1, sagittal T2/STIR, axial T2) |
| Output | One of three grades for each of the 25 condition-level labels |
| Computational task | 25 three-class classification problems per study |

Keep three units apart throughout this lecture and the lab. A **study** is one MRI visit. A **slice** (DICOM *instance*) is one 2D image. A **label** is one condition-level grade. This study has 96 slices but 25 labels.

## 2. MRI in brief

### 2.1 How MRI makes an image

**Magnetic resonance imaging (MRI)** places the body in a strong magnetic field and sends radio-frequency pulses. Hydrogen nuclei, mostly in water and fat, respond and emit signals. Different tissues return to equilibrium ("relax") at different rates, and this difference creates image contrast. MRI uses no ionizing radiation.

A scan acquires a stack of thin 2D **slices**. One run of one pulse sequence in one orientation is a **series**.

### 2.2 Weightings: why the same anatomy looks different

By changing the timing of the pulses, the scanner emphasizes different tissue properties. This is called the **weighting**.

| Sequence | Fluid (CSF, healthy disc) | Fat | Used in this dataset for |
|---|---|---|---|
| **T1-weighted** | Dark | Bright | Neural foramina: the bright fat around each nerve root makes narrowing visible |
| **T2-weighted** | Bright | Intermediate to bright | Spinal canal and lateral recesses: bright CSF outlines the nerve roots |
| **STIR** (short tau inversion recovery) | Bright | Suppressed (dark) | Fluid-sensitive images; the dataset accepts STIR as a "T2-like" sagittal series |

Healthy discs contain water and look bright on T2. Degenerated discs lose water and look darker.

### 2.3 Planes

| Plane | Viewpoint | In this study |
|---|---|---|
| **Sagittal** | Side view, slicing left to right | Sagittal T1 and Sagittal T2/STIR, 18 slices each |
| **Axial** | Cross-section, slicing head to feet | Axial T2, 60 slices |
| **Coronal** | Front view | Not used here |

### 2.4 MRI numbers have no fixed unit

In CT, pixel values are calibrated to **Hounsfield units**, where water is 0 by definition. MRI intensities have **no such fixed scale**. The same tissue can have very different values on different scanners, coils, or settings.

This matters for AI. A model trained at one hospital can meet different intensity distributions at another, which is a form of **dataset shift**. Normalizing each image helps, but it does not remove differences in contrast, noise, or protocol.

## 3. DICOM: how medical images are stored

### 3.1 What DICOM is

**DICOM** (Digital Imaging and Communications in Medicine) is the international standard for medical images. It defines both a file format and a network protocol. Scanners send DICOM objects to a **PACS** (picture archiving and communication system), where radiologists view them.

Each DICOM file has two parts:

1. a **header**: a list of **data elements**, each identified by a tag `(group, element)`, a keyword, and a value;
2. the **pixel data**: the image itself, often with 12–16 bits per pixel.

### 3.2 The hierarchy

| Level | Meaning | DICOM identifier | In the RSNA files |
|---|---|---|---|
| Patient | The person | `PatientID (0010,0020)` | Replaced by the study number |
| **Study** | One imaging visit | `StudyInstanceUID (0020,000D)` | `study_id` column; top folder |
| **Series** | One sequence in one plane | `SeriesInstanceUID (0020,000E)` | `series_id` column; sub-folder |
| **Instance** | One image (slice) | `SOPInstanceUID (0008,0018)`, `InstanceNumber (0020,0013)` | `<instance_number>.dcm` |

### 3.3 Identifiers are not content

A **UID** (unique identifier) tells a computer which object it is handling. It says nothing about anatomy or disease.

Headers can also hold personal information such as names, birth dates, and hospital numbers. Text can even be "burned into" the pixels. **De-identification** removes or replaces such data before research use. The RSNA contributing sites de-identified their data, and many tags (for example `Modality` and the patient's name) are absent from these files.

### 3.4 DICOM compared with PNG and JPEG

| Property | PNG / JPEG | DICOM |
|---|---|---|
| Typical bit depth | 8 bits (256 grey levels) | 12–16 bits (here 12 bits: 4,096 levels) |
| Metadata | Minimal | Rich: identifiers, acquisition settings, geometry |
| Physical scale | Pixels only | Millimetres via `PixelSpacing` and positions |
| Grouping | Single picture | Instance → series → study |
| Clinical use | Exports and teaching | Standard for clinical storage and exchange (DICOM files can themselves use JPEG-family compression) |

Let us read the header of one axial slice. pydicom reads it as a `Dataset`, whose elements we can access by keyword.

In [3]:
example_path = IMAGE_ROOT / "1012284084" / "46.dcm"   # one axial T2 slice
example = pydicom.dcmread(example_path)

# Show every header element except the large pixel block.
header_rows = [
    {"tag": str(elem.tag), "keyword": elem.keyword, "value": str(elem.value)[:60]}
    for elem in example
    if elem.keyword != "PixelData"
]
display(pd.DataFrame(header_rows))
print("Pixel array:", example.pixel_array.shape, example.pixel_array.dtype,
      "| min", example.pixel_array.min(), "| max", example.pixel_array.max())

,tag,keyword,value
0,"(0008,0018)",SOPInstanceUID,100206310.1.46
1,"(0008,0023)",ContentDate,20240503
2,"(0008,0033)",ContentTime,223741.892820
3,"(0008,103E)",SeriesDescription,T2
4,"(0010,0020)",PatientID,100206310
5,"(0018,0050)",SliceThickness,3.5
6,"(0018,0088)",SpacingBetweenSlices,3.5
7,"(0018,5100)",PatientPosition,HFS
8,"(0020,000D)",StudyInstanceUID,100206310
9,"(0020,000E)",SeriesInstanceUID,100206310.1012284084


Pixel array: (320, 320) uint16 | min 0 | max 1259


The header has only 26 elements because the files were de-identified. `SeriesDescription` says only `T2`. The fuller description ("Axial T2") comes from the dataset's CSV file, a reminder that metadata can live outside the image.

The pixel values are integers. With `BitsStored = 12`, they could range from 0 to 4,095, but this slice uses only part of that range.

## 4. Geometry: from pixels to millimetres and patient directions

### 4.1 Size

- `PixelSpacing` gives the physical size of one pixel in mm (row spacing, column spacing).
- `SliceThickness` is the thickness of the tissue slab captured by one slice.
- `SpacingBetweenSlices` is the distance between slice centres. If it is larger than the thickness, there are small gaps.

**Worked example.** The axial slice above is 320 × 320 pixels with 0.5 mm spacing. The **field of view** is 320 × 0.5 = **160 mm** on each side. The sagittal T1 series has 512 pixels at 0.684 mm, giving about **350 mm**: enough to show the whole lumbar spine.

### 4.2 Direction: the patient coordinate system

DICOM places every image in a 3D coordinate system attached to the patient, in millimetres ([DICOM PS3.3 §C.7.6.2.1.1](https://dicom.nema.org/medical/dicom/current/output/chtml/part03/sect_C.7.6.2.html)):

| Axis | Increases towards |
|---|---|
| +x | the patient's **left** |
| +y | the patient's **posterior** (back) |
| +z | the patient's **head** (superior) |

This is often called **LPS** (left, posterior, superior).

- `ImagePositionPatient` = the (x, y, z) position in mm of the centre of the top-left pixel.
- `ImageOrientationPatient` = six numbers. The first three are the direction you move in when going **along a row** (left to right on screen). The last three are the direction you move in when going **down a column**.

The cross product of the two directions gives the **slice normal**: the direction perpendicular to the image. Its largest component tells us the plane.

**Worked example.** For an axial slice, the row direction is about (1, 0, 0) and the column direction about (0, 1, 0). Their cross product is (0, 0, 1), which points along z (head–feet), so the slices are stacked from head to feet: **axial**. A normal along x (left–right) means **sagittal**.

### 4.3 Left and right on screen

A row direction of +x means that moving right across the screen moves towards the **patient's left**. That is the **radiological convention**: an axial image is viewed as if looking up from the patient's feet, so the patient's left appears on the viewer's right. Getting this wrong swaps left and right labels, a classic and dangerous error.

In [4]:
row_dir = np.array(example.ImageOrientationPatient[:3], dtype=float)
col_dir = np.array(example.ImageOrientationPatient[3:], dtype=float)
normal = np.cross(row_dir, col_dir)

print("Row direction   :", row_dir.round(3), "→ moving right on screen moves towards the patient's",
      "LEFT" if row_dir[0] > 0 else "RIGHT")
print("Column direction:", col_dir.round(3))
print("Slice normal    :", normal.round(3), "→ largest component on axis",
      "xyz"[int(np.argmax(np.abs(normal)))])
print("Field of view   :", example.Rows * float(example.PixelSpacing[0]), "mm ×",
      example.Columns * float(example.PixelSpacing[1]), "mm")

Row direction   : [ 0.999  0.03  -0.042] → moving right on screen moves towards the patient's LEFT
Column direction: [-0.02   0.976  0.216]
Slice normal    : [ 0.048 -0.215  0.975] → largest component on axis z
Field of view   : 160.0 mm × 160.0 mm


The normal is not exactly (0, 0, 1). This axial series is acquired in tilted blocks so that slices lie roughly parallel to each disc, and the lower discs are tilted more. You will see this in [Lab 9](../../labs/lab-09-lumbar-mri-dicom-medgemma.ipynb).

## 5. Displaying MRI: window/level, and what a model actually sees

### 5.1 Window/level

A screen shows about 256 grey levels. A 12-bit image has up to 4,096. **Window/level** chooses which range of raw values to stretch across the screen:

$$
I_{\text{display}} = 255 \times \frac{\operatorname{clip}(I_{\text{raw}},\ L - W/2,\ L + W/2) - (L - W/2)}{W}
$$

- $I_{\text{raw}}$ — the stored pixel value (arbitrary MRI units);
- $L$ — the **level** (window centre): the middle of the range of interest;
- $W$ — the **window width**: how wide that range is;
- `clip` sets values below $L - W/2$ to that lower bound (black) and values above $L + W/2$ to the upper bound (white).

**Worked example.** The header of the axial slice we opened (instance 46) suggests $L = 576$ and $W = 1235$. The displayed range is $576 - 617.5 = -41.5$ to $576 + 617.5 = 1193.5$. A raw value of 600 becomes $255 \times (600 + 41.5) / 1235 \approx 132$, a mid grey. Every raw value above 1193.5 becomes pure white, so differences between those bright pixels are lost on screen.

A narrow window increases contrast within a small range. A wide window shows more of the range with less contrast.

### 5.2 MRI has no universal presets

CT viewers have standard presets such as "bone" or "lung" because Hounsfield units are fixed. MRI has none. The `WindowCenter`/`WindowWidth` values in the header are suggestions from the scanner or technologist. A common alternative is a **percentile window**, for example the 1st to 99th percentile of the slice's own values.

### 5.3 What MedGemma receives

To send a slice to MedGemma we:

1. apply a window (12-bit → 8-bit), discarding grey-level detail;
2. save it as a PNG picture of one 2D slice;
3. let the model resize it: MedGemma's image encoder normalizes images to 896 × 896 pixels and turns each image into 256 tokens ([MedGemma model card](https://developers.google.com/health-ai-developer-foundations/medgemma/model-card)).

A radiologist scrolls through every slice of three series at full bit depth. The model in our setup sees **one compressed 2D picture per request**. Keep that gap in mind when interpreting its answers.

## 6. Two ways to build imaging AI

| | **Task-specific model** | **Foundation vision–language model** |
|---|---|---|
| Example | A convolutional neural network (CNN) trained on thousands of labelled lumbar MRIs, as in the Kaggle competition | MedGemma, prompted with an image and instructions |
| How it learns this task | Supervised training on this exact label set | Broad pretraining and medical adaptation; no training on our labels (**zero-shot**) |
| Output | Calibrated class probabilities for fixed labels | Free text, which we must parse into labels |
| Main strengths | Usually stronger on the narrow task it was trained for | Flexible; no task-specific training data needed |
| Main risks | Needs large labelled data; fails under dataset shift | Fluent but unsupported answers; format errors; untested on many tasks |

### 6.1 MedGemma in one paragraph

**MedGemma** is a collection of Google models built on **Gemma 3**, a decoder-only language model, with a **SigLIP-based image encoder** pretrained on de-identified medical data. **MedGemma 1.5 4B** (released 13 January 2026) adds support for 3D CT and MRI volumes, among other image types. Details of the architecture are in the [foundation models lecture](../../../module2/lectures/session-05-medical-foundation-models.ipynb).

The model card sets clear limits ([MedGemma model card](https://developers.google.com/health-ai-developer-foundations/medgemma/model-card), checked 9 October 2026):

> "MedGemma's multimodal capabilities have been primarily evaluated on single-image tasks."
>
> "MedGemma has not been evaluated in use cases that involve comprehension of multiple images."
>
> "The outputs generated by MedGemma are not intended to directly inform clinical diagnosis, patient management decisions, treatment recommendations, or any other direct clinical practice applications."

### 6.2 Our setup is not the reported model

The saved responses used in Lab 9 come from `medgemma-1.5-4b-it` served by LM Studio as a **4-bit quantized GGUF** file (`Q4_K_S`). Quantization stores the weights at lower precision to save memory and can change outputs. We also send **2D PNG slices through a chat API**, not 3D volumes prepared as described by the model's developers. Our results therefore describe **this setup on this patient**. They do not measure MedGemma's 3D MRI capability or its published benchmark performance.

## 7. How we will evaluate MedGemma

These are the experiments you run in [Lab 9](../../labs/lab-09-lumbar-mri-dicom-medgemma.ipynb). Here we fix the design and the scoring rules before seeing any results.

### 7.1 Four experiments

| ID | Experiment | Images sent | What it tests |
|---|---|---|---|
| **E0** | Sanity check | One slice from each series, separately | Can the model name the modality, body region, and plane? We can check this against the DICOM geometry. |
| **E1** | Naive whole-study prompt | The middle slice of each series, together in one request | All 25 grades at once from arbitrary slices. This is how a first attempt often looks. |
| **E2** | Expert-guided prompts | The slice where the expert placed each annotation; one condition per request | Best-case input: the right slice, the right view, a narrow question |
| **E3** | Stress tests | Repeated, blank, and mirrored images | Consistency, refusal to answer without evidence, and left/right reasoning |

**A warning about E2.** E2 chooses slices using the expert's annotation coordinates, which are part of the reference standard. A deployed system would not have them. E2 is a best-case probe of the model, not a usable pipeline, and its scores are optimistic.

### 7.2 Metrics

- **Format compliance**: how many of the 25 labels came back as an allowed grade? "Cannot assess" is allowed and counted separately; anything else is invalid.
- **Exact agreement**: labels where the model grade equals the reference grade, out of all 25.
- **High-grade detection**: treat **Moderate or Severe** as "positive" (clinically relevant narrowing). Then compute sensitivity, specificity, positive predictive value (PPV), and negative predictive value (NPV) on the labels the model actually graded.
- **Baseline**: always answer "Normal/Mild", the most common class in the dataset. A model must beat this trivial rule to show it is reading the image.

**Worked calculation (hypothetical model).** Suppose 16 of 25 reference labels are positive and 9 are negative, and a model calls 10 labels positive, of which 8 are truly positive.

| | Reference positive | Reference negative |
|---|---|---|
| Model positive | TP = 8 | FP = 2 |
| Model negative | FN = 8 | TN = 7 |

- Sensitivity = TP / (TP + FN) = 8/16 = **50%**: half the narrowed sites were found.
- Specificity = TN / (TN + FP) = 7/9 = **78%**.
- PPV = TP / (TP + FP) = 8/10 = **80%**: when it says "narrowed", it is usually right.
- NPV = TN / (TN + FN) = 7/15 = **47%**: a "normal" answer is wrong more than half the time.

### 7.3 Why one patient cannot establish performance

- The 25 labels are **not independent**: they share one person's anatomy, one scanner, and one image quality.
- The reference is **one annotator's** reading.
- One patient covers a tiny part of the variety of ages, diseases, scanners, and sites.
- A confidence interval computed as if the 25 labels were independent would be falsely narrow, so we do not report one.

A real evaluation would need hundreds of held-out studies, ideally from sites not used in development, a test set with multiple annotators, prespecified metrics, subgroup analysis, and a study of how radiologists actually use the output.

### Review questions

1. Why is "grading stenosis on MRI" a narrower task than "diagnosing the cause of back pain"?
2. Which series would you look at to grade left foraminal narrowing at L4/L5, and why does T1 weighting help?
3. A file is named `20.dcm`. Which DICOM tag, not the file name, should decide its order in the series?
4. A slice has `ImageOrientationPatient = [0, 1, 0, 0, 0, -1]`. What is its plane?
5. Why can't you reuse a CT "bone window" on MRI?
6. Give two reasons why MedGemma's published benchmark scores do not predict its accuracy in Lab 9.
7. Why are the E2 results optimistic?

## Key ideas

- The clinical task is **anatomical grading**, not diagnosing pain. Define the user, the unit of analysis (study, slice, label), and the decision time before choosing a model.
- **DICOM** carries both pixels and geometry. Order slices by tags, not file names, and use `ImageOrientationPatient` and `ImagePositionPatient` to know where left, right, up, and down are.
- **Window/level** changes what is visible, and the 8-bit PNG sent to a model inherits that choice.
- A vision–language model's fluent answer is not evidence. Ask for **structured output**, fix the scoring rules first, compare with a **baseline**, and **stress-test** the behaviour.
- One patient, one annotator, and a quantized local model can **describe** behaviour and expose failure modes. They cannot establish clinical performance.

## References and further reading

- Richards TJ, Flanders AE, Colak E, et al. [The RSNA Lumbar Degenerative Imaging Spine Classification (LumbarDISC) Dataset](https://pubs.rsna.org/doi/10.1148/ryai.250480). *Radiology: Artificial Intelligence* 2026;8(2):e250480. Preprint: [arXiv:2506.09162](https://arxiv.org/abs/2506.09162). Dataset composition, annotation process, label distribution.
- RSNA. [RSNA 2024 Lumbar Spine Degenerative Classification](https://www.kaggle.com/competitions/rsna-2024-lumbar-spine-degenerative-classification) (Kaggle competition, data, and rules).
- Google. [MedGemma model card](https://developers.google.com/health-ai-developer-foundations/medgemma/model-card) and [Health AI Developer Foundations terms of use](https://developers.google.com/health-ai-developer-foundations/terms). Checked 9 October 2026.
- NEMA. [DICOM PS3.3, §C.7.6.2 Image Plane Module](https://dicom.nema.org/medical/dicom/current/output/chtml/part03/sect_C.7.6.2.html): definitions of image position, orientation, and the patient coordinate system.
- [pydicom documentation](https://pydicom.github.io/pydicom/stable/): reading DICOM files in Python.
- Brinjikji W, Luetmer PH, Comstock B, et al. [Systematic literature review of imaging features of spinal degeneration in asymptomatic populations](https://www.ajnr.org/content/36/4/811). *AJNR* 2015;36(4):811–816.

*Optional further study:* the [foundation models lecture](../../../module2/lectures/session-05-medical-foundation-models.ipynb) covers MedGemma's architecture and evidence-grounded evaluation in more depth.